In [1]:
import pandas as pd
import math
from collections import Counter

In [2]:
df = pd.read_csv(
    "../../dataset/experiment/all_similarity_output.csv"
)

df

,news_id,similar_news_id,score
0,1,2,0.022751
1,1,3,0.003745
2,1,4,0.013939
3,1,5,0.010383
4,1,6,0.010593
...,...,...,...
9895,100,95,0.015278
9896,100,96,0.013406
9897,100,97,0.004191
9898,100,98,0.018260


In [3]:
candidate_df = df[
    df["news_id"] == 1
]

candidate_df

,news_id,similar_news_id,score
0,1,2,0.022751
1,1,3,0.003745
2,1,4,0.013939
3,1,5,0.010383
4,1,6,0.010593
...,...,...,...
94,1,96,0.017611
95,1,97,0.021079
96,1,98,0.022414
97,1,99,0.010125


In [4]:
candidate_df = candidate_df.sort_values(
    by="score",
    ascending=False
)

candidate_df

,news_id,similar_news_id,score
56,1,58,0.109693
41,1,43,0.074443
37,1,39,0.072643
79,1,81,0.067864
18,1,20,0.067517
...,...,...,...
1,1,3,0.003745
40,1,42,0.003340
58,1,60,0.002948
90,1,92,0.002454


In [5]:
candidate_df = candidate_df.head(10)

candidate_df

,news_id,similar_news_id,score
56,1,58,0.109693
41,1,43,0.074443
37,1,39,0.072643
79,1,81,0.067864
18,1,20,0.067517
21,1,23,0.047382
85,1,87,0.046844
80,1,82,0.045867
5,1,7,0.044706
69,1,71,0.044254


In [7]:
feedback_data = {
    58: [1, 1, 0, 1, 1],
    43: [1, 0, 1],
    39: [1, 1, 1, 0],
    81: [0, 1],
    20: [1, 0, 0],
    23: [1, 1, 0, 1],
    87: [0, 0, 1],
    82: [1, 1],
    7: [1, 0, 1, 1],
    71: [0, 1, 0]
}

In [6]:
class UcbTunedNotebookService:

    def _mean_reward(self, rewards):
        if not rewards:
            return 0.0

        return sum(rewards) / len(rewards)


    def _variance_vj(self, rewards, t):
        s = len(rewards)

        if s == 0:
            return 0.0

        t = max(t, 2)

        mean = self._mean_reward(rewards)
        mean_square = sum(x**2 for x in rewards) / s

        return mean_square - (mean ** 2) + math.sqrt((2 * math.log(t)) / s)


    def _ucb_tuned_score(self, rewards, t):
        s = len(rewards)

        if s == 0:
            return 0.0

        t = max(t, 2)

        mean = self._mean_reward(rewards)
        vj = self._variance_vj(rewards, t)

        bonus = math.sqrt((math.log(t) / s) * min(0.25, vj))

        return mean + bonus


    def recommend(self, candidate_df, feedback_data, top_k=5):
        t = sum(len(rewards) for rewards in feedback_data.values())
        t = max(t, 2)

        ranked = []

        for _, item in candidate_df.iterrows():
            nid = int(item["similar_news_id"])
            raw_rewards = feedback_data.get(nid, [])
            s = len(raw_rewards)

            if s == 0:
                ranked.append({
                    "news_id": nid,
                    "cbf_score": float(item["score"]),
                    "ucb_score": 0.0,
                    "mean_reward": 0.0,
                    "views": 0,
                    "clicks": 0,
                    "V_j": 0.0
                })
                continue

            mean = self._mean_reward(raw_rewards)
            vj = self._variance_vj(raw_rewards, t)
            ucb_score = self._ucb_tuned_score(raw_rewards, t)

            ranked.append({
                "news_id": nid,
                "cbf_score": float(item["score"]),
                "ucb_score": float(ucb_score),
                "mean_reward": float(mean),
                "views": s,
                "clicks": sum(raw_rewards),
                "V_j": float(vj)
            })

        ranked = sorted(
            ranked,
            key=lambda x: x["ucb_score"],
            reverse=True
        )

        return {
            "recommendations": ranked[:top_k],
            "all_ranked": ranked
        }

In [12]:
ucb_service = UcbTunedNotebookService()

result = ucb_service.recommend(
    candidate_df=candidate_df,
    feedback_data=feedback_data,
    top_k=5
)

final_df = pd.DataFrame({
    "news_id": [1] * len(result["recommendations"]),
    "similar_news_id": [
        item["news_id"]
        for item in result["recommendations"]
    ],
    "ucb_score": [
        item["ucb_score"]
        for item in result["recommendations"]
    ]
})

final_df

,news_id,similar_news_id,ucb_score
0,1,82,1.661108
1,1,58,1.218121
2,1,39,1.217474
3,1,23,1.217474
4,1,7,1.217474


In [ ]:
# ucb_result_df.to_csv(
#     "../../dataset/experiment/ucb_tuned_result.csv",
#     index=False
# )